In [1]:
#1.Loader
from langchain_community.document_loaders import DirectoryLoader, TextLoader
#2. Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
#3. embeddings
from langchain_ollama import OllamaEmbeddings
#4 . vecotor store
from langchain_community.vectorstores import FAISS


In [3]:
#load documents
loader = DirectoryLoader(
    path="../data/physics",
    glob="*.txt",
    loader_cls=TextLoader
)
documents = loader.load()

In [12]:
#invetigate documents
print("document :",documents[0])
print("Meta Data: ",documents[0].metadata)

document : page_content='CONCEPT 4: ACCELERATION

Acceleration describes how quickly velocity changes with time. An object accelerates when it speeds up, slows down, or changes direction. Acceleration is present in many daily activities, such as starting a bike, applying brakes, or turning a vehicle. Even objects moving at constant speed can have acceleration if their direction changes.

The concept of acceleration helps explain why forces are needed to change motion. It is closely related to Newtonâ€™s laws of motion and gravity. At the Class 10 level, acceleration is important for understanding free fall, vehicle safety, and motion graphs.

1. Acceleration is the rate of change of velocity.
2. It tells how quickly velocity changes with time.
3. Acceleration occurs when speed changes.
4. Acceleration also occurs when direction changes.
5. Even at constant speed, acceleration can exist.
6. Circular motion has acceleration due to change in direction.
7. The SI unit of acceleration is me

In [15]:
#chuncking
text_spliter = RecursiveCharacterTextSplitter(
    chunk_size= 100,
    chunk_overlap=20,
)

In [17]:
chunks = text_spliter.split_documents(documents)
chunks[0].metadata

{'source': '..\\data\\physics\\accelaration.txt'}

In [21]:
#embedings
embeddings = OllamaEmbeddings(
    model="embeddinggemma",
)

In [23]:
#store in vectore store
vector_store = FAISS.from_documents(chunks,embedding= embeddings)

In [24]:
# create as retriver
retriver = vector_store.as_retriever()

In [25]:
#search query
question = "what is gravity?"

In [29]:
#simentic searchs
relevant_data = retriver.invoke(question)
print(relevant_data)
print(relevant_data[0].metadata)
print(len(relevant_data))

[Document(id='8c1fce93-9a9c-41d2-aa28-87fda1988b29', metadata={'source': '..\\data\\physics\\force.txt'}, page_content='23. Example: A book resting on a table.\n24. Gravity pulls the book downward.'), Document(id='77baad76-dc2d-4e5e-9f9f-5fd8a0c68696', metadata={'source': '..\\data\\physics\\newtonslaws.txt'}, page_content='These laws help explain why objects remain at rest, why heavier objects need more force to move,'), Document(id='9cc7e50d-4a1a-44f1-9fbe-a897037e72bc', metadata={'source': '..\\data\\physics\\newtonslaws.txt'}, page_content='1. Newtonâ€™s laws describe the relationship between force and motion.'), Document(id='698b300a-7a1d-49d3-904b-da7476d8a895', metadata={'source': '..\\data\\physics\\accelaration.txt'}, page_content='32. Falling objects accelerate due to gravity.\n33. Gravity causes constant acceleration.')]
{'source': '..\\data\\physics\\force.txt'}
4


In [36]:
# create a prompt template
from langchain_core.prompts import ChatPromptTemplate
teacher_template = ChatPromptTemplate.from_messages(
            [
            ("system", """
             you are a helpful {class} class subject {subject} teacher.
             you explain concepts in simpe terms.
             you answe question based on the context provided.
             if you don't know the answer, just say 'I dont'n know'.
             Don't make up answers.
             context is {context}
             """),
            ("human", "{question}")
        ]
    
)

In [ ]:
# sample invocation of template
# teacher_template.invoke(
#     {
#         "class": "10",
#         "subject": "science",
#         "context": relevant_data,
#         "question": question,
#     }
# )

# print(teacher_template)

input_variables=['class', 'context', 'question', 'subject'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['class', 'context', 'subject'], input_types={}, partial_variables={}, template="\n             you are a helpful {class} class subject {subject} teacher.\n             you explain concepts in simpe terms.\n             you answe question based on the context provided.\n             if you don't know the answer, just say 'I dont'n know'.\n             Don't make up answers.\n             context is {context}\n             "), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]


In [41]:
#LLM
from langchain_ollama.llms import OllamaLLM
model = OllamaLLM(model="gemma3:1b")

In [44]:
#createa langchain
rag_chain = teacher_template | model

In [46]:
response = rag_chain.invoke(
    {
        "class": "10",
        "subject": "science",
        "context": relevant_data,
        "question": question,
    }
)
response

'Gravity is a force that pulls things together. It’s what keeps your feet on the ground!\n\n'